# 📚 BioRAG — Notebook 4: Q&A with Claude
### Literature-Grounded AI Answers with Exact Passage Citations

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YOUR_USERNAME/genesight-biorag/blob/main/notebooks/04_BioRAG_QA_Demo.ipynb)

---

## What this notebook does

Connects the ChromaDB vector store from Notebook 3 to the Claude API to build a full
**Retrieval-Augmented Generation (RAG)** Q&A system for your own biomedical paper library.

**Two use cases demonstrated:**

### 1 · Peer Review Assistant
You are reviewing a manuscript. You have downloaded the cited papers.
You upload them to BioRAG and ask precise, methodological questions:
- *"What methylation array QC thresholds are used in these papers?"*
- *"How do these authors define differentially methylated regions?"*
- *"Is the statistical approach in these papers consistent?"*
BioRAG retrieves the exact passages from your papers and Claude synthesizes
a cited answer — all locally, with nothing transmitted externally.

### 2 · GeneSight Literature Integration
A variant was flagged by GeneSight with a high SHAP contribution from `log_af_genome`.
You ask BioRAG: *"What does the literature say about allele frequency as a predictor
of pathogenicity in pediatric cancer variants?"*
BioRAG pulls the relevant passages from your paper library and Claude explains the
biological rationale grounded in your own references.

> **Privacy:** Your PDFs are embedded locally. ChromaDB stores only vector representations.
> No manuscript text is sent to any external service. The Claude API receives only
> retrieved passage snippets (≤ 3,000 tokens of context), not your full PDFs.


## ⚙️ Setup

In [ ]:
!pip install anthropic chromadb sentence-transformers pypdf pandas --quiet
print("✅ Packages installed")

In [ ]:
import os
import json
import textwrap
from pathlib import Path
from typing import List, Dict, Optional
from IPython.display import Markdown, display

import chromadb
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer

# Anthropic client — API key loaded from Colab Secrets (see Step 1)
import anthropic

# ── Paths (must match Notebook 3) ─────────────────────────────────────────────
DB_DIR  = Path("biorag_db")
DATA_DIR = Path("data")

# ── Constants (must match Notebook 3) ─────────────────────────────────────────
MODEL_NAME       = "pritamdeka/BioBERT-mnli-snli-scinli-scitail-mednli-stsb"
COLLECTION_NAME  = "biorag_papers"
CLAUDE_MODEL     = "claude-sonnet-4-20250514"
MAX_TOKENS       = 1500
TOP_K_RETRIEVAL  = 5     # passages retrieved per query
MAX_CONTEXT_CHARS = 3000  # character cap on context sent to Claude

print("✅ Imports complete")

---
## 🔑 Step 1: Configure Anthropic API Key

**How to add your key in Colab:**
1. Click the 🔑 **Secrets** icon in the left sidebar
2. Click **+ Add new secret**
3. Name: `ANTHROPIC_API_KEY`  Value: your API key (starts with `sk-ant-...`)
4. Toggle **Notebook access** ON
5. Re-run this cell


In [ ]:
try:
    from google.colab import userdata
    ANTHROPIC_API_KEY = userdata.get('ANTHROPIC_API_KEY')
    print("✅ API key loaded from Colab Secrets")
except Exception:
    # Local development fallback
    ANTHROPIC_API_KEY = os.environ.get("ANTHROPIC_API_KEY", "")
    if ANTHROPIC_API_KEY:
        print("✅ API key loaded from environment variable")
    else:
        print("⚠️  No API key found. Set ANTHROPIC_API_KEY in Colab Secrets.")

client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)
print(f"   Using model: {CLAUDE_MODEL}")

---
## 🗄️ Step 2: Load ChromaDB + Embedding Model from Notebook 3

In [ ]:
# ── Embedding model (same as Notebook 3 — must match for retrieval to work) ───
print(f"🤖 Loading embedding model: {MODEL_NAME}")
print("   (Cached after first download)")
embedder = SentenceTransformer(MODEL_NAME)
print(f"✅ Embedder loaded | dim={embedder.get_sentence_embedding_dimension()}")

# ── ChromaDB ──────────────────────────────────────────────────────────────────
if not DB_DIR.exists():
    print(f"⚠️  {DB_DIR}/ not found — run Notebook 3 first to build the vector store.")
else:
    chroma_client = chromadb.PersistentClient(path=str(DB_DIR))
    try:
        collection = chroma_client.get_collection(COLLECTION_NAME)
        print(f"✅ Loaded collection '{COLLECTION_NAME}': {collection.count():,} chunks")
    except Exception as e:
        print(f"❌ Could not load collection: {e}")
        print("   Run Notebook 3 first to build the ChromaDB index.")

# ── Show indexed papers ───────────────────────────────────────────────────────
manifest_path = DB_DIR / "index_manifest.json"
if manifest_path.exists():
    with open(manifest_path) as f:
        manifest = json.load(f)
    print(f"\n📋 {len(manifest)} papers indexed:")
    for m in manifest:
        print(f"   • {m['title'][:70]}  [{m['num_chunks']} chunks]")
else:
    print("\n   (No manifest found — run Notebook 3 to generate one)")

---
## 🧠 Step 3: The BioRAG Q&A Engine

In [ ]:
def retrieve(query: str, top_k: int = TOP_K_RETRIEVAL) -> List[Dict]:
    """
    Embed query and retrieve the top-k most relevant passages from ChromaDB.
    Returns list of dicts with text, metadata, and similarity score.
    """
    q_emb = embedder.encode(
        [query], normalize_embeddings=True, convert_to_numpy=True
    )
    results = collection.query(
        query_embeddings=q_emb.tolist(),
        n_results=min(top_k, collection.count()),
        include=['documents', 'metadatas', 'distances']
    )
    passages = []
    for doc, meta, dist in zip(
        results['documents'][0],
        results['metadatas'][0],
        results['distances'][0]
    ):
        passages.append({
            'text':       doc,
            'filename':   meta.get('filename', ''),
            'title':      meta.get('title', meta.get('filename', '')),
            'author':     meta.get('author', ''),
            'chunk_idx':  meta.get('chunk_idx', 0),
            'similarity': round(1 - dist, 4),
        })
    return passages


def build_context(passages: List[Dict], max_chars: int = MAX_CONTEXT_CHARS) -> str:
    """
    Format retrieved passages as numbered context blocks for Claude.
    Truncates to max_chars to stay within a safe token budget.
    """
    blocks, total = [], 0
    for i, p in enumerate(passages, 1):
        label  = f"[{i}] {p['title']} (similarity: {p['similarity']:.3f})"
        block  = f"{label}\n{p['text'].strip()}"
        if total + len(block) > max_chars:
            break
        blocks.append(block)
        total += len(block)
    return "\n\n---\n\n".join(blocks)


def ask_biorag(
    question: str,
    top_k: int = TOP_K_RETRIEVAL,
    system_persona: str = "peer_review",
    verbose: bool = True,
) -> Dict:
    """
    Full BioRAG pipeline: retrieve → build context → call Claude → return answer.

    Parameters
    ----------
    question      : Natural language question about your papers
    top_k         : Number of passages to retrieve
    system_persona: 'peer_review' or 'research'
    verbose       : Print formatted answer to cell output

    Returns
    -------
    dict with keys: question, answer, passages, tokens_used
    """
    # ── 1. Retrieve ────────────────────────────────────────────────────────────
    passages = retrieve(question, top_k=top_k)
    context  = build_context(passages)

    # ── 2. System prompt — two personas ───────────────────────────────────────
    if system_persona == "peer_review":
        system = """You are a meticulous biomedical peer reviewer helping evaluate
a manuscript under review. Your task is to answer questions about the
scientific literature based ONLY on the retrieved passages provided.

Rules:
- Ground every claim in a specific passage; cite it as [N] after each claim.
- If the passages do not contain enough information to answer, say so explicitly.
- Be concise and precise — this is for scientific review, not explanation.
- Flag any inconsistencies you notice across the provided sources.
- Never fabricate citations or data not present in the passages.
- End with a one-sentence 'Reviewer note' summarizing the key finding."""
    else:
        system = """You are an expert bioinformatics research assistant.
Answer the question based ONLY on the retrieved passages.
Cite each claim with [N] referring to the passage number.
If the passages are insufficient, say so and suggest what additional
literature might help. Be scientifically precise."""

    # ── 3. User message ────────────────────────────────────────────────────────
    user_msg = f"""Question: {question}

Retrieved passages from your paper library:
{context}

Answer the question based on these passages. Cite each claim with [N]."""

    # ── 4. Call Claude ─────────────────────────────────────────────────────────
    response = client.messages.create(
        model      = CLAUDE_MODEL,
        max_tokens = MAX_TOKENS,
        system     = system,
        messages   = [{"role": "user", "content": user_msg}],
    )
    answer      = response.content[0].text
    tokens_used = response.usage.input_tokens + response.usage.output_tokens

    # ── 5. Format output ───────────────────────────────────────────────────────
    if verbose:
        display(Markdown(f"### ❓ {question}"))
        display(Markdown("---"))
        display(Markdown(answer))
        display(Markdown("---"))
        display(Markdown("**Sources retrieved:**"))
        for i, p in enumerate(passages, 1):
            sim_bar = "█" * int(p['similarity'] * 10) + "░" * (10 - int(p['similarity'] * 10))
            display(Markdown(
                f"`[{i}]` **{p['filename']}** "
                f"(similarity: {p['similarity']:.3f} {sim_bar})"
            ))
        display(Markdown(f"\n*Tokens used: {tokens_used:,}*"))

    return {
        'question':    question,
        'answer':      answer,
        'passages':    passages,
        'tokens_used': tokens_used,
    }

print("✅ BioRAG Q&A engine ready")
print(f"   Retrieval: top-{TOP_K_RETRIEVAL} passages, max {MAX_CONTEXT_CHARS} chars context")
print(f"   LLM: {CLAUDE_MODEL}, max {MAX_TOKENS} output tokens")

---
## 📋 Step 4: Peer Review Demo — Methodological Questions

These are the kinds of questions you ask when reviewing a manuscript.
Each answer includes exact passage citations and a reviewer note.

> **How this helps your review:**
> Instead of re-reading every cited paper's methods section, BioRAG finds
> the relevant passages in seconds and synthesizes them with citations
> you can verify in the originals. Your review stays rigorous; it just takes
> a fraction of the time.


In [ ]:
# ── Run each query with peer_review persona ──────────────────────────────────
# ✏️  Edit these questions to match the manuscript you are reviewing.

peer_review_queries = [
    "What methylation array QC thresholds are used for IDAT array data in these papers?",
    "How do these papers define differentially methylated regions (DMRs)?",
    "What normalization methods are applied to methylation beta values before analysis?",
]

peer_review_results = []
for q in peer_review_queries:
    print("=" * 70)
    result = ask_biorag(q, system_persona="peer_review")
    peer_review_results.append(result)
    print()

---
## ✏️ Step 5: Ask Your Own Peer Review Question

Replace the string below with any question about your loaded papers.


In [ ]:
# ── Your custom question ─────────────────────────────────────────────────────
my_question = """
What statistical methods are used to assess differential methylation,
and are the sample sizes sufficient for the reported effect sizes?
"""

# Change system_persona to 'research' for a less reviewer-specific tone
my_result = ask_biorag(my_question.strip(), system_persona="peer_review")

---
## 🧬 Step 6: GeneSight × BioRAG Integration

When GeneSight flags a variant, the top SHAP feature tells you *which signal
drove the prediction*. You can feed that directly into BioRAG to get a
literature-grounded explanation of the biological rationale.

**Workflow:**
1. GeneSight predicts a variant as Pathogenic (confidence: 0.91)
2. SHAP shows `log_af_genome` was the dominant feature
3. BioRAG retrieves what your papers say about allele frequency and pathogenicity
4. Claude synthesizes a cited explanation you can include in your analysis notes

> This is the "Both" demo from the presentation — a full end-to-end loop.


In [ ]:
# ── Load GeneSight predictions if available ───────────────────────────────────
preds_path = DATA_DIR / "genesight_test_predictions.csv"

if preds_path.exists():
    preds = pd.read_csv(preds_path)
    # Pick the highest-confidence Pathogenic prediction
    example = preds[preds['true_label'] == 1].sort_values('pred_proba', ascending=False).iloc[0]
    gene        = example['GeneSymbol']
    chrom       = example['Chromosome']
    pos         = int(example['Start'])
    proba       = example['pred_proba']
    top_feature = example['top_shap_feature']
    top_shap    = example['top_shap_value']
    print(f"📌 GeneSight example variant:")
    print(f"   Gene:           {gene}  (chr{chrom}:{pos:,})")
    print(f"   P(Pathogenic):  {proba:.3f}")
    print(f"   Top SHAP feat:  {top_feature}  (value: {top_shap:+.4f})")
else:
    # Demo values if Notebook 2 hasn't been run yet
    gene, chrom, pos = "BRCA1", "17", 43044295
    proba, top_feature, top_shap = 0.912, "log_af_genome", -3.21
    print("📌 Demo variant (run Notebook 2 to load real GeneSight predictions):")
    print(f"   Gene: {gene}  P(Pathogenic): {proba:.3f}  Top feature: {top_feature}")

In [ ]:
# ── Build a context-aware literature query from the SHAP output ───────────────
feature_descriptions = {
    'log_af_genome':   'population allele frequency in genome cohorts',
    'log_af_exome':    'population allele frequency in exome cohorts',
    'pLI':             'probability of loss-of-function intolerance (pLI)',
    'LOEUF':           'loss-of-function observed/expected upper bound (LOEUF)',
    'is_lof':          'loss-of-function consequence (nonsense or frameshift)',
    'is_missense':     'missense amino acid change consequence',
    'is_nonsense':     'nonsense (stop-gain) consequence',
    'is_splice':       'splice site disruption consequence',
    'is_frameshift':   'frameshift insertion or deletion consequence',
    'review_confidence':'ClinVar review status confidence level',
    'num_submitters':  'number of ClinVar submitters',
    'chrom_encoded':   'chromosomal location encoding',
    'is_synonymous':   'synonymous (silent) mutation consequence',
    'is_transition':   'transition mutation type (A↔G or C↔T)',
}
feat_desc = feature_descriptions.get(top_feature, top_feature)

integration_query = (
    f"What does the literature say about {feat_desc} "
    f"as a predictor of variant pathogenicity, "
    f"particularly in the context of {gene} or pediatric cancer variants?"
)

print(f"🔍 Auto-generated literature query:")
print(f"   {integration_query}")
print()

integration_result = ask_biorag(integration_query, system_persona="research")

---
## 📦 Step 7: Batch Mode — Multiple Queries at Once

Useful when reviewing a manuscript with several methodological sections
or when you want to systematically document your literature search.


In [ ]:
def batch_biorag(queries: List[str], persona: str = "peer_review") -> pd.DataFrame:
    """
    Run multiple BioRAG queries and return results as a DataFrame.
    Useful for exporting your literature review notes.
    """
    records = []
    for i, q in enumerate(queries, 1):
        print(f"Query {i}/{len(queries)}: {q[:60]}...")
        result = ask_biorag(q, system_persona=persona, verbose=False)
        # Summarise top source
        top_src = result['passages'][0]['filename'] if result['passages'] else "None"
        top_sim = result['passages'][0]['similarity'] if result['passages'] else 0.0
        records.append({
            'query':           q,
            'answer_excerpt':  result['answer'][:300] + "...",
            'top_source':      top_src,
            'top_similarity':  round(top_sim, 3),
            'tokens_used':     result['tokens_used'],
        })
        print(f"   ✅ Top source: {top_src}  (sim={top_sim:.3f})")

    return pd.DataFrame(records)


# ── Example batch — edit these queries for your manuscript ────────────────────
batch_queries = [
    "What array platforms and probe counts are used in the methylation studies?",
    "How are CpG islands defined and filtered in these analyses?",
    "What bioinformatics pipelines (e.g. minfi, ChAMP) are referenced?",
]

batch_df = batch_biorag(batch_queries)
print("\n📊 Batch results summary:")
display(batch_df[['query', 'top_source', 'top_similarity', 'tokens_used']])

# Save to CSV for your review notes
batch_df.to_csv(DATA_DIR / "biorag_review_notes.csv", index=False)
print(f"\n✅ Saved to {DATA_DIR}/biorag_review_notes.csv")

---
## 📄 Step 8: Export Full Review Notes

Saves all your Q&A results to a Markdown file you can attach to your review notes
or paste into your review submission.


In [ ]:
def export_review_notes(results: List[Dict], output_path: Path) -> None:
    """Export Q&A results to a clean Markdown file."""
    lines = [
        "# BioRAG Literature Review Notes\n",
        f"*Generated from {collection.count():,} indexed chunks*\n",
        "---\n",
    ]
    for i, r in enumerate(results, 1):
        lines.append(f"## Query {i}: {r['question']}\n")
        lines.append(r['answer'] + "\n\n")
        lines.append("**Sources:**\n")
        for j, p in enumerate(r['passages'], 1):
            lines.append(
                f"- [{j}] {p['filename']}  "
                f"(similarity: {p['similarity']:.3f})\n"
            )
        lines.append("\n---\n")

    with open(output_path, 'w') as f:
        f.writelines(lines)
    print(f"✅ Review notes saved to {output_path}")


# Collect all results from Steps 4 and 5
all_results = peer_review_results + [my_result, integration_result]
export_review_notes(all_results, DATA_DIR / "biorag_review_notes.md")
print("\nYou can download this file from the Colab file browser (left sidebar → Files)")

---
## ✅ Summary

In this notebook we:

1. **Configured** the Anthropic API key securely via Colab Secrets
2. **Loaded** the BioBERT embedding model and ChromaDB vector store from Notebook 3
3. **Built** the `ask_biorag()` function — retrieve → context → Claude → cited answer
4. **Demonstrated peer review use case** — methodological Q&A with reviewer-tone answers
5. **Demonstrated GeneSight integration** — SHAP feature → literature query → cited explanation
6. **Batch mode** — multiple queries, CSV export for review documentation
7. **Exported** full review notes to Markdown

### Privacy reminder
Your PDFs are embedded locally. Claude only receives:
- Your question (typed by you)
- ≤ 3,000 characters of retrieved passage snippets

No full PDF text, no manuscript titles, no author names are sent to Anthropic.

---
## Architecture recap

```
Your question
     ↓
BioBERT embedding (local, in Colab)
     ↓
ChromaDB vector search (local, on disk)
     ↓
Top-K passage snippets (text only, ≤3,000 chars)
     ↓
Claude API (receives only snippets + your question)
     ↓
Cited answer with [N] references
```

---
*BioRAG is an open-source research tool.
Always verify AI-synthesized answers against the original source documents.*
